In [1]:
import os
import logging
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("pipeline.log", mode="w"),
        logging.StreamHandler()
    ]
)

logging.info("Starting the traffic data pipeline.")

def load_csv_data(file_name):
    logging.info("Attempting to load the CSV file.")
    if not os.path.exists(file_name):
        logging.error("Error: The CSV file was not found in this folder.")
        return None
    try:
        data = pd.read_csv(file_name)
        logging.info("Success: CSV file loaded into memory.")
        return data
    except pd.errors.ParserError:
        logging.error("Error: The CSV file formatting is corrupted.")
        return None
    except Exception as e:
        logging.error(f"An unexpected error happened: {e}")
        return None

def validate_schema(df):
    if df is None:
        return False
    logging.info("Validating the data schema columns.")
    expected_columns = [
        'holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 
        'weather_main', 'weather_description', 'date_time', 'traffic_volume'
    ]
    missing_columns = []
    for col in expected_columns:
        if col not in df.columns:
            missing_columns.append(col)
    if len(missing_columns) > 0:
        logging.error(f"Schema Validation Failed! Missing columns: {missing_columns}")
        return False
    logging.info("Schema Validation Successful: All expected columns are present.")
    return True

def clean_data(df):
    if df is None:
        return None
        
    logging.info("Starting data cleaning process.")
    
    initial_rows = len(df)
    df = df.drop_duplicates()
    duplicate_count = initial_rows - len(df)
    logging.info(f"Removed {duplicate_count} duplicate rows.")
    
    df['weather_main'] = df['weather_main'].str.strip().str.title()
    df['weather_description'] = df['weather_description'].str.strip().str.lower()
    df['holiday'] = df['holiday'].str.strip()
    logging.info("Standardised categorical text values.")
    
    try:
        df['date_time'] = pd.to_datetime(df['date_time'])
        logging.info("Parsed and validated date/time fields.")
    except Exception as e:
        logging.error(f"Date/time parsing failed: {e}")
        
    median_temp = df.loc[df['temp'] > 0, 'temp'].median()
    zero_temp_mask = df['temp'] == 0
    zero_temp_count = zero_temp_mask.sum()
    if zero_temp_count > 0:
        df.loc[zero_temp_mask, 'temp'] = median_temp
        logging.info(f"Handled temperature outliers: Imputed {zero_temp_count} instances of 0 Kelvin using median ({median_temp}K).")
        
    median_rain = df.loc[df['rain_1h'] < 9000, 'rain_1h'].median()
    extreme_rain_mask = df['rain_1h'] >= 9000
    extreme_rain_count = extreme_rain_mask.sum()
    if extreme_rain_count > 0:
        df.loc[extreme_rain_mask, 'rain_1h'] = median_rain
        logging.info(f"Handled rainfall outliers: Imputed {extreme_rain_count} impossible values exceeding 9K mm using median ({median_rain}mm).")
        
    return df

def clean_and_transform(df):
    if df is None:
        logging.warning("No data found to process. Stopping here.")
        return None
    logging.info("Starting data checks.")
    missing_data = df.isnull().sum()
    logging.info("Checking for missing values in each column:")
    for column, count in missing_data.items():
        if count > 0:
            logging.warning(f"Column '{column}' has {count} missing rows.")
    logging.info("Checking column data types.")
    logging.info("Extracting hour from date_time column.")
    try:
        df['hour'] = df['date_time'].dt.hour
    except Exception as e:
        logging.error(f"Could not extract hour dimension: {e}")
    logging.info("Converting temperature from Kelvin to Celsius.")
    df['temp_celsius'] = df['temp'] - 273.15
    logging.info("Creating traffic category column.")
    def check_volume(volume):
        if volume > 5500:
            return 'High'
        elif volume >= 2500:
            return 'Medium'
        else:
            return 'Low'
    df['traffic_category'] = df['traffic_volume'].apply(check_volume)
    logging.info("All data pipeline transformations completed successfully.")
    return df

if __name__ == "__main__":
    my_file = "Metro_Interstate_Traffic_Volume.csv"
    raw_df = load_csv_data(my_file)
    if validate_schema(raw_df):
        cleaned_df = clean_data(raw_df)
        final_df = clean_and_transform(cleaned_df)
    else:
        logging.error("Pipeline stopped because schema validation failed.")


2026-09-24 16:42:27,595 - INFO - Starting the traffic data pipeline.
2026-09-24 16:42:27,597 - INFO - Attempting to load the CSV file.
2026-09-24 16:42:27,638 - INFO - Success: CSV file loaded into memory.
2026-09-24 16:42:27,638 - INFO - Validating the data schema columns.
2026-09-24 16:42:27,639 - INFO - Schema Validation Successful: All expected columns are present.
2026-09-24 16:42:27,639 - INFO - Starting data cleaning process.
2026-09-24 16:42:27,657 - INFO - Removed 17 duplicate rows.
C:\Users\User\AppData\Local\Temp\ipykernel_28008\3071710836.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['weather_main'] = df['weather_main'].str.strip().str.title()
C:\Users\User\AppData\Local\Temp\ipykernel_28008\3071710836.py:62: SettingWithCopy